# Calculus for Machine Learning

## Overview
Calculus is essential for understanding how machine learning algorithms learn. This notebook covers:
- Derivatives and rates of change
- Partial derivatives
- Gradients and gradient descent
- Chain rule and backpropagation
- Optimization fundamentals

**Time to complete:** 3-4 hours  
**Prerequisites:** Basic algebra, linear algebra basics

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
from matplotlib import cm

np.random.seed(42)

## 1. Derivatives: Rate of Change

The derivative measures how a function changes as its input changes.

**Definition:**
$$f'(x) = \lim_{h \to 0} \frac{f(x+h) - f(x)}{h}$$

**Interpretation:** The slope of the tangent line at point $x$

### Common Derivatives
- $\frac{d}{dx}(x^n) = nx^{n-1}$ (Power rule)
- $\frac{d}{dx}(e^x) = e^x$ (Exponential)
- $\frac{d}{dx}(\ln x) = \frac{1}{x}$ (Natural log)
- $\frac{d}{dx}(\sin x) = \cos x$ (Sine)
- $\frac{d}{dx}(\cos x) = -\sin x$ (Cosine)

In [ ]:
# Numerical approximation of derivative
def numerical_derivative(f, x, h=1e-5):
    """Compute derivative using finite difference."""
    return (f(x + h) - f(x)) / h

# Example: f(x) = x^2
f = lambda x: x**2
f_prime_analytical = lambda x: 2*x  # Analytical derivative

x = 3
numerical = numerical_derivative(f, x)
analytical = f_prime_analytical(x)

print(f"f(x) = x^2 at x = {x}")
print(f"Numerical derivative: {numerical:.6f}")
print(f"Analytical derivative: {analytical:.6f}")
print(f"Error: {abs(numerical - analytical):.10f}")

In [ ]:
# Visualize function and its derivative
x = np.linspace(-3, 3, 100)
y = x**2
y_prime = 2*x

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))

# Plot function
ax1.plot(x, y, 'b-', linewidth=2, label='f(x) = x²')
ax1.set_xlabel('x')
ax1.set_ylabel('f(x)')
ax1.set_title('Function')
ax1.grid(True, alpha=0.3)
ax1.legend()

# Plot derivative
ax2.plot(x, y_prime, 'r-', linewidth=2, label="f'(x) = 2x")
ax2.axhline(y=0, color='k', linestyle='--', alpha=0.3)
ax2.set_xlabel('x')
ax2.set_ylabel("f'(x)")
ax2.set_title('Derivative (Slope)')
ax2.grid(True, alpha=0.3)
ax2.legend()

plt.tight_layout()
plt.show()

print("Notice:")
print("- When x < 0: derivative is negative (function decreasing)")
print("- When x = 0: derivative is zero (local minimum)")
print("- When x > 0: derivative is positive (function increasing)")

In [ ]:
# Tangent line visualization
def plot_tangent(f, f_prime, x0):
    """Plot function with tangent line at x0."""
    x = np.linspace(x0-2, x0+2, 100)
    y = f(x)
    
    # Tangent line: y = f(x0) + f'(x0)(x - x0)
    y0 = f(x0)
    slope = f_prime(x0)
    tangent = y0 + slope * (x - x0)
    
    plt.figure(figsize=(10, 6))
    plt.plot(x, y, 'b-', linewidth=2, label='f(x)')
    plt.plot(x, tangent, 'r--', linewidth=2, label=f'Tangent at x={x0}')
    plt.plot(x0, y0, 'go', markersize=10, label=f'Point ({x0}, {y0:.2f})')
    plt.xlabel('x')
    plt.ylabel('y')
    plt.title(f'Tangent Line (slope = {slope:.2f})')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.show()

# Example: f(x) = x^3 - 3x
f = lambda x: x**3 - 3*x
f_prime = lambda x: 3*x**2 - 3

plot_tangent(f, f_prime, x0=1)

## 2. Rules of Differentiation

### Sum Rule
$$\frac{d}{dx}[f(x) + g(x)] = f'(x) + g'(x)$$

### Product Rule
$$\frac{d}{dx}[f(x) \cdot g(x)] = f'(x)g(x) + f(x)g'(x)$$

### Chain Rule (Most Important for ML!)
$$\frac{d}{dx}[f(g(x))] = f'(g(x)) \cdot g'(x)$$

The chain rule is the foundation of backpropagation in neural networks!

In [ ]:
# Chain rule example: f(g(x)) where g(x) = x^2 and f(u) = e^u
# So f(g(x)) = e^(x^2)

# Analytical derivative using chain rule:
# d/dx[e^(x^2)] = e^(x^2) * 2x

def composite_function(x):
    return np.exp(x**2)

def composite_derivative(x):
    return np.exp(x**2) * 2*x

x = 1.5
numerical = numerical_derivative(composite_function, x)
analytical = composite_derivative(x)

print("Chain Rule Example: f(g(x)) = e^(x²)")
print(f"\nAt x = {x}:")
print(f"Numerical derivative: {numerical:.6f}")
print(f"Analytical derivative (chain rule): {analytical:.6f}")
print(f"Error: {abs(numerical - analytical):.10f}")

print("\nChain rule breakdown:")
print(f"g(x) = x² → g'(x) = 2x = {2*x}")
print(f"f(u) = e^u → f'(u) = e^u = e^({x**2}) = {np.exp(x**2):.4f}")
print(f"f'(g(x)) * g'(x) = {np.exp(x**2):.4f} * {2*x} = {analytical:.4f}")

## 3. Partial Derivatives

For functions with multiple variables: $f(x, y)$

**Partial derivative** with respect to $x$:
$$\frac{\partial f}{\partial x} = \lim_{h \to 0} \frac{f(x+h, y) - f(x, y)}{h}$$

Hold all other variables constant, differentiate with respect to one variable.

**Example:** $f(x, y) = x^2 + 3xy + y^2$
- $\frac{\partial f}{\partial x} = 2x + 3y$
- $\frac{\partial f}{\partial y} = 3x + 2y$

In [ ]:
# Partial derivatives example
def f(x, y):
    """f(x, y) = x^2 + 3xy + y^2"""
    return x**2 + 3*x*y + y**2

def partial_x(x, y):
    """∂f/∂x = 2x + 3y"""
    return 2*x + 3*y

def partial_y(x, y):
    """∂f/∂y = 3x + 2y"""
    return 3*x + 2*y

# Numerical partial derivatives
def numerical_partial_x(f, x, y, h=1e-5):
    return (f(x + h, y) - f(x, y)) / h

def numerical_partial_y(f, x, y, h=1e-5):
    return (f(x, y + h) - f(x, y)) / h

x, y = 2, 3
print(f"f(x, y) = x² + 3xy + y² at ({x}, {y})")
print(f"f({x}, {y}) = {f(x, y)}")

print(f"\n∂f/∂x:")
print(f"  Analytical: {partial_x(x, y)}")
print(f"  Numerical:  {numerical_partial_x(f, x, y):.6f}")

print(f"\n∂f/∂y:")
print(f"  Analytical: {partial_y(x, y)}")
print(f"  Numerical:  {numerical_partial_y(f, x, y):.6f}")

In [ ]:
# Visualize function and partial derivatives
x = np.linspace(-3, 3, 50)
y = np.linspace(-3, 3, 50)
X, Y = np.meshgrid(x, y)
Z = X**2 + 3*X*Y + Y**2

fig = plt.figure(figsize=(15, 5))

# 3D surface
ax1 = fig.add_subplot(131, projection='3d')
surf = ax1.plot_surface(X, Y, Z, cmap=cm.viridis, alpha=0.8)
ax1.set_xlabel('x')
ax1.set_ylabel('y')
ax1.set_zlabel('f(x, y)')
ax1.set_title('Function f(x, y)')

# Contour plot
ax2 = fig.add_subplot(132)
contour = ax2.contour(X, Y, Z, levels=20)
ax2.clabel(contour, inline=True, fontsize=8)
ax2.set_xlabel('x')
ax2.set_ylabel('y')
ax2.set_title('Contour Plot')
ax2.grid(True, alpha=0.3)

# Gradient field
ax3 = fig.add_subplot(133)
# Compute gradients
dfdx = 2*X + 3*Y
dfdy = 3*X + 2*Y
# Normalize for visualization
magnitude = np.sqrt(dfdx**2 + dfdy**2)
dfdx_norm = dfdx / (magnitude + 1e-8)
dfdy_norm = dfdy / (magnitude + 1e-8)
# Plot every 5th arrow
skip = 5
ax3.quiver(X[::skip, ::skip], Y[::skip, ::skip], 
           dfdx_norm[::skip, ::skip], dfdy_norm[::skip, ::skip])
ax3.set_xlabel('x')
ax3.set_ylabel('y')
ax3.set_title('Gradient Field')
ax3.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 4. Gradients

The **gradient** is a vector of all partial derivatives:

$$\nabla f = \begin{bmatrix} \frac{\partial f}{\partial x_1} \\ \frac{\partial f}{\partial x_2} \\ \vdots \\ \frac{\partial f}{\partial x_n} \end{bmatrix}$$

**Key property:** The gradient points in the direction of steepest ascent!

**For ML:** We use the negative gradient to minimize loss functions (gradient descent)

In [ ]:
# Gradient computation
def compute_gradient(f, x, h=1e-5):
    """Compute gradient numerically for multi-variable function."""
    n = len(x)
    grad = np.zeros(n)
    for i in range(n):
        x_plus = x.copy()
        x_plus[i] += h
        grad[i] = (f(x_plus) - f(x)) / h
    return grad

# Example: f(x, y) = x^2 + y^2 (paraboloid)
def paraboloid(x):
    return x[0]**2 + x[1]**2

# Analytical gradient: [2x, 2y]
def paraboloid_gradient(x):
    return np.array([2*x[0], 2*x[1]])

point = np.array([3.0, 4.0])
numerical_grad = compute_gradient(paraboloid, point)
analytical_grad = paraboloid_gradient(point)

print(f"Point: {point}")
print(f"Function value: {paraboloid(point)}")
print(f"\nNumerical gradient: {numerical_grad}")
print(f"Analytical gradient: {analytical_grad}")
print(f"Error: {np.linalg.norm(numerical_grad - analytical_grad)}")

## 5. Gradient Descent

**Goal:** Minimize a function $f(\mathbf{x})$

**Algorithm:**
1. Start with initial guess $\mathbf{x}_0$
2. Compute gradient $\nabla f(\mathbf{x}_t)$
3. Update: $\mathbf{x}_{t+1} = \mathbf{x}_t - \alpha \nabla f(\mathbf{x}_t)$
4. Repeat until convergence

Where $\alpha$ is the **learning rate** (step size)

In [ ]:
# Gradient descent implementation
def gradient_descent(f, grad_f, x0, learning_rate=0.1, n_iterations=100, tolerance=1e-6):
    """Perform gradient descent optimization."""
    x = x0.copy()
    history = [x.copy()]
    
    for i in range(n_iterations):
        grad = grad_f(x)
        x_new = x - learning_rate * grad
        
        # Check convergence
        if np.linalg.norm(x_new - x) < tolerance:
            print(f"Converged after {i+1} iterations")
            break
        
        x = x_new
        history.append(x.copy())
    
    return x, np.array(history)

# Minimize f(x, y) = x^2 + y^2
x0 = np.array([5.0, 5.0])
optimal_x, history = gradient_descent(paraboloid, paraboloid_gradient, x0, 
                                      learning_rate=0.1, n_iterations=50)

print(f"\nStarting point: {x0}")
print(f"Optimal point: {optimal_x}")
print(f"Function value at optimal: {paraboloid(optimal_x):.10f}")
print(f"True minimum: [0, 0] with value 0")

In [ ]:
# Visualize gradient descent path
x = np.linspace(-6, 6, 100)
y = np.linspace(-6, 6, 100)
X, Y = np.meshgrid(x, y)
Z = X**2 + Y**2

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# Contour plot with path
contour = ax1.contour(X, Y, Z, levels=30, cmap='viridis')
ax1.clabel(contour, inline=True, fontsize=8)
ax1.plot(history[:, 0], history[:, 1], 'r.-', linewidth=2, markersize=8, label='GD path')
ax1.plot(history[0, 0], history[0, 1], 'go', markersize=12, label='Start')
ax1.plot(history[-1, 0], history[-1, 1], 'r*', markersize=15, label='End')
ax1.set_xlabel('x')
ax1.set_ylabel('y')
ax1.set_title('Gradient Descent Path')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Function value over iterations
values = [paraboloid(h) for h in history]
ax2.plot(values, 'b-', linewidth=2)
ax2.set_xlabel('Iteration')
ax2.set_ylabel('Function Value')
ax2.set_title('Convergence')
ax2.grid(True, alpha=0.3)
ax2.set_yscale('log')

plt.tight_layout()
plt.show()

In [ ]:
# Effect of learning rate
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
learning_rates = [0.01, 0.1, 0.5, 0.9]

for idx, lr in enumerate(learning_rates):
    ax = axes[idx // 2, idx % 2]
    
    # Run gradient descent
    x0 = np.array([5.0, 5.0])
    optimal_x, history = gradient_descent(paraboloid, paraboloid_gradient, x0,
                                         learning_rate=lr, n_iterations=50)
    
    # Plot
    x = np.linspace(-6, 6, 100)
    y = np.linspace(-6, 6, 100)
    X, Y = np.meshgrid(x, y)
    Z = X**2 + Y**2
    
    contour = ax.contour(X, Y, Z, levels=20, cmap='viridis', alpha=0.5)
    ax.plot(history[:, 0], history[:, 1], 'r.-', linewidth=2, markersize=6)
    ax.plot(history[0, 0], history[0, 1], 'go', markersize=10)
    ax.plot(history[-1, 0], history[-1, 1], 'r*', markersize=12)
    ax.set_xlabel('x')
    ax.set_ylabel('y')
    ax.set_title(f'Learning Rate = {lr} ({len(history)} iterations)')
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("Observations:")
print("- Too small LR: slow convergence")
print("- Good LR: fast, smooth convergence")
print("- Too large LR: oscillations, may diverge")

## 6. Chain Rule and Backpropagation

Neural networks are compositions of functions:
$$y = f_3(f_2(f_1(x)))$$

To compute $\frac{dy}{dx}$, we use the chain rule:
$$\frac{dy}{dx} = \frac{dy}{df_3} \cdot \frac{df_3}{df_2} \cdot \frac{df_2}{df_1} \cdot \frac{df_1}{dx}$$

This is **backpropagation**: computing gradients by going backwards through the network!

In [ ]:
# Simple neural network example: y = f(w2 * f(w1 * x))
# where f is activation function (ReLU)

def relu(x):
    """ReLU activation."""
    return np.maximum(0, x)

def relu_derivative(x):
    """Derivative of ReLU."""
    return (x > 0).astype(float)

# Forward pass
def forward(x, w1, w2):
    """Two-layer network forward pass."""
    z1 = w1 * x
    a1 = relu(z1)
    z2 = w2 * a1
    a2 = relu(z2)
    return a2, (x, z1, a1, z2)

# Backward pass (backpropagation)
def backward(dy, cache, w1, w2):
    """Compute gradients using backpropagation."""
    x, z1, a1, z2 = cache
    
    # Gradient at output
    dz2 = dy * relu_derivative(z2)
    dw2 = dz2 * a1
    da1 = dz2 * w2
    
    # Gradient at hidden layer
    dz1 = da1 * relu_derivative(z1)
    dw1 = dz1 * x
    
    return dw1, dw2

# Example
x = 2.0
w1 = 0.5
w2 = 1.5

# Forward pass
y, cache = forward(x, w1, w2)
print(f"Input: x = {x}")
print(f"Weights: w1 = {w1}, w2 = {w2}")
print(f"Output: y = {y}")

# Backward pass (assume dy/dy = 1)
dw1, dw2 = backward(1.0, cache, w1, w2)
print(f"\nGradients:")
print(f"dL/dw1 = {dw1}")
print(f"dL/dw2 = {dw2}")

# Verify with numerical gradient
def loss(w1, w2):
    y, _ = forward(x, w1, w2)
    return y

h = 1e-5
numerical_dw1 = (loss(w1 + h, w2) - loss(w1, w2)) / h
numerical_dw2 = (loss(w1, w2 + h) - loss(w1, w2)) / h

print(f"\nNumerical gradients:")
print(f"dL/dw1 = {numerical_dw1:.6f}")
print(f"dL/dw2 = {numerical_dw2:.6f}")
print(f"\nErrors:")
print(f"w1 error: {abs(dw1 - numerical_dw1):.10f}")
print(f"w2 error: {abs(dw2 - numerical_dw2):.10f}")

## 7. Common Activation Functions and Derivatives

### Sigmoid
$$\sigma(x) = \frac{1}{1 + e^{-x}}$$
$$\sigma'(x) = \sigma(x)(1 - \sigma(x))$$

### Tanh
$$\tanh(x) = \frac{e^x - e^{-x}}{e^x + e^{-x}}$$
$$\tanh'(x) = 1 - \tanh^2(x)$$

### ReLU
$$\text{ReLU}(x) = \max(0, x)$$
$$\text{ReLU}'(x) = \begin{cases} 1 & x > 0 \\ 0 & x \leq 0 \end{cases}$$

In [ ]:
# Activation functions and their derivatives
x = np.linspace(-5, 5, 100)

# Sigmoid
sigmoid = 1 / (1 + np.exp(-x))
sigmoid_derivative = sigmoid * (1 - sigmoid)

# Tanh
tanh = np.tanh(x)
tanh_derivative = 1 - tanh**2

# ReLU
relu = np.maximum(0, x)
relu_derivative = (x > 0).astype(float)

# Plot
fig, axes = plt.subplots(3, 2, figsize=(15, 12))

# Sigmoid
axes[0, 0].plot(x, sigmoid, 'b-', linewidth=2)
axes[0, 0].set_title('Sigmoid: σ(x)')
axes[0, 0].grid(True, alpha=0.3)
axes[0, 0].set_ylabel('σ(x)')

axes[0, 1].plot(x, sigmoid_derivative, 'r-', linewidth=2)
axes[0, 1].set_title("Sigmoid Derivative: σ'(x)")
axes[0, 1].grid(True, alpha=0.3)
axes[0, 1].set_ylabel("σ'(x)")

# Tanh
axes[1, 0].plot(x, tanh, 'b-', linewidth=2)
axes[1, 0].set_title('Tanh: tanh(x)')
axes[1, 0].grid(True, alpha=0.3)
axes[1, 0].set_ylabel('tanh(x)')

axes[1, 1].plot(x, tanh_derivative, 'r-', linewidth=2)
axes[1, 1].set_title("Tanh Derivative: tanh'(x)")
axes[1, 1].grid(True, alpha=0.3)
axes[1, 1].set_ylabel("tanh'(x)")

# ReLU
axes[2, 0].plot(x, relu, 'b-', linewidth=2)
axes[2, 0].set_title('ReLU: max(0, x)')
axes[2, 0].grid(True, alpha=0.3)
axes[2, 0].set_xlabel('x')
axes[2, 0].set_ylabel('ReLU(x)')

axes[2, 1].plot(x, relu_derivative, 'r-', linewidth=2)
axes[2, 1].set_title("ReLU Derivative: ReLU'(x)")
axes[2, 1].grid(True, alpha=0.3)
axes[2, 1].set_xlabel('x')
axes[2, 1].set_ylabel("ReLU'(x)")

plt.tight_layout()
plt.show()

## 8. Key Takeaways

1. **Derivatives** measure rate of change
   - Used to find optimal parameters
   - Direction of steepest descent

2. **Partial derivatives** for multi-variable functions
   - Each parameter has its own gradient
   - Combined into gradient vector

3. **Gradient descent** is the optimization workhorse
   - Update: $\theta_{t+1} = \theta_t - \alpha \nabla L(\theta_t)$
   - Learning rate is crucial

4. **Chain rule** enables backpropagation
   - Compute gradients efficiently
   - Foundation of deep learning

5. **Activation functions** introduce non-linearity
   - Must be differentiable (almost everywhere)
   - ReLU is most common in deep learning

## Practice Problems

1. Compute derivatives of: $f(x) = 3x^4 - 2x^3 + 5x - 1$
2. Find critical points of: $f(x, y) = x^2 + y^2 - 2x - 4y + 5$
3. Implement gradient descent for: $f(x) = x^4 - 3x^3 + 2$
4. Derive backpropagation for 3-layer network
5. Implement momentum-based gradient descent
6. Compare different learning rate schedules

## Next Steps
- Complete exercises in `exercises.md`
- Work on the mini project: Matrix Calculator
- Move to Week 2: NumPy and Pandas